In [1]:
import os
import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import precision_score, recall_score, f1_score

def set_seeds(seed: int = 42):
    """Sets random seeds for reproducible runs."""
    np.random.seed(seed)
    tf.random.set_seed(seed)

def load_and_preprocess_data(file_path: str, num_clients: int = 20):
    """
    Loads raw UCI dataset using semicolon separators and European comma decimals.
    Resamples 15-min intervals to hourly sums, selects top clients with early reporting,
    trims leading zero periods, and constructs cyclical time features.
    """
    df = pd.read_csv(
        file_path,
        sep=';',
        decimal=',',
        index_col=0,
        parse_dates=True,
        low_memory=False
    )
    df.index.name = 'timestamp'

    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # Hourly resampling by summation
    df_hourly = df.resample('h').sum().dropna(how='all')

    # Select 20 clients with complete data history (earliest first non-zero timestamp)
    first_nonzero_idx = (df_hourly != 0).idxmax()
    selected_clients = first_nonzero_idx.sort_values().index[:num_clients].tolist()

    # Trim leading zeros distortion
    start_timestamp = first_nonzero_idx[selected_clients].max()
    df_trimmed = df_hourly.loc[start_timestamp:, selected_clients].copy()

    # Cyclical Sine/Cosine Encodings for hour of day and day of week
    df_trimmed['hour_sin'] = np.sin(2 * np.pi * df_trimmed.index.hour / 24.0)
    df_trimmed['hour_cos'] = np.cos(2 * np.pi * df_trimmed.index.hour / 24.0)
    df_trimmed['day_sin'] = np.sin(2 * np.pi * df_trimmed.index.dayofweek / 7.0)
    df_trimmed['day_cos'] = np.cos(2 * np.pi * df_trimmed.index.dayofweek / 7.0)

    return df_trimmed, selected_clients

def split_and_scale(df: pd.DataFrame, client_cols: list, train_ratio=0.70, val_ratio=0.15):
    """
    Splits data chronologically (70% Train, 15% Val, 15% Test) before scaling
    to prevent data leakage. MinMaxScaler is fitted strictly on the training set.
    """
    n = len(df)
    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))

    df_train = df.iloc[:train_end].copy()
    df_val = df.iloc[train_end:val_end].copy()
    df_test = df.iloc[val_end:].copy()

    scaler = MinMaxScaler(feature_range=(0, 1))
    scaler.fit(df_train[client_cols])

    df_train[client_cols] = scaler.transform(df_train[client_cols])
    df_val[client_cols] = scaler.transform(df_val[client_cols])
    df_test[client_cols] = scaler.transform(df_test[client_cols])

    return df_train, df_val, df_test, scaler

def create_sequences(df: pd.DataFrame, client_cols: list, history_size=24, future_target=24):
    """Creates 24-hour lookback sequences (X) and 24-hour forecast target sequences (y)."""
    feature_data = df.values
    client_indices = [df.columns.get_loc(c) for c in client_cols]

    xs, ys = [], []
    total_len = len(df) - history_size - future_target + 1

    for i in range(total_len):
        xs.append(feature_data[i : i + history_size])
        ys.append(feature_data[i + history_size : i + history_size + future_target, client_indices])

    return np.array(xs), np.array(ys)

def build_lstm_attention_model(input_shape, output_steps=24, num_targets=20, d_model=64, num_heads=4, ff_dim=128):
    """
    LSTM Encoder followed by Multi-Head Self-Attention over sequence timesteps.
    Setting return_sequences=True allows attention to compare across all 24 sequence steps.
    """
    inputs = layers.Input(shape=input_shape)
    x = layers.LSTM(d_model, return_sequences=True)(inputs)

    # Multi-Head Attention over timesteps
    attn_output = layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model)(x, x)
    attn_output = layers.Dropout(0.1)(attn_output)
    x = layers.LayerNormalization(epsilon=1e-6)(x + attn_output)

    # Feed-Forward Network
    ffn = layers.Dense(ff_dim, activation="relu")(x)
    ffn = layers.Dense(d_model)(ffn)
    ffn = layers.Dropout(0.1)(ffn)
    x = layers.LayerNormalization(epsilon=1e-6)(x + ffn)

    # Forecast projection head
    x = layers.Flatten()(x)
    x = layers.Dense(output_steps * num_targets)(x)
    outputs = layers.Reshape((output_steps, num_targets))(x)

    model = Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])
    return model

def calculate_metrics(y_true: np.ndarray, y_pred: np.ndarray):
    """Calculates MAE, RMSE, and sMAPE cleanly without division-by-zero warnings."""
    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))

    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    with np.errstate(divide='ignore', invalid='ignore'):
        smape_elements = np.divide(np.abs(y_true - y_pred), denom)
        smape_elements[denom == 0] = 0.0
    smape = np.mean(smape_elements) * 100.0

    return {'MAE': float(mae), 'RMSE': float(rmse), 'sMAPE': float(smape)}

def inverse_transform_targets(y_scaled: np.ndarray, scaler, num_clients: int):
    """Reshapes and unscales arrays back to original kWh/kW values."""
    shape = y_scaled.shape
    flat = y_scaled.reshape(-1, num_clients)
    unscaled = scaler.inverse_transform(flat)
    return unscaled.reshape(shape)

def main():
    set_seeds(42)
    artifacts_dir = 'artifacts'
    os.makedirs(artifacts_dir, exist_ok=True)

    # Explicit path to Google Colab root environment
    file_path = '/content/drive/MyDrive/energydiagram/LD2011_2014.txt'
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File '{file_path}' not found. Please upload LD2011_2014.txt to Google Colab.")

    print("\n[1/5] Loading and Preprocessing Electricity Dataset...")
    df, client_cols = load_and_preprocess_data(file_path, num_clients=20)
    df_train, df_val, df_test, scaler = split_and_scale(df, client_cols)

    print("\n[2/5] Creating Sliding Window Sequences (24h Input -> 24h Horizon)...")
    X_train, y_train = create_sequences(df_train, client_cols)
    X_val, y_val = create_sequences(df_val, client_cols)
    X_test, y_test = create_sequences(df_test, client_cols)

    print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
    print(f"X_test shape:  {X_test.shape}, y_test shape:  {y_test.shape}")

    # Save artifacts for dashboard
    df_test_unscaled = df_test.copy()
    df_test_unscaled[client_cols] = scaler.inverse_transform(df_test[client_cols])
    joblib.dump(df_test_unscaled.head(500), os.path.join(artifacts_dir, 'sample_df.pkl'))
    joblib.dump(scaler, os.path.join(artifacts_dir, 'scaler.pkl'))
    joblib.dump(client_cols, os.path.join(artifacts_dir, 'client_cols.pkl'))

    print("\n[3/5] Building & Training LSTM + Multi-Head Attention Model...")
    input_shape = (X_train.shape[1], X_train.shape[2])
    model = build_lstm_attention_model(input_shape, output_steps=24, num_targets=len(client_cols))

    callbacks = [EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)]
    model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=30,
        batch_size=64,
        callbacks=callbacks,
        verbose=1
    )

    model.save(os.path.join(artifacts_dir, 'model.keras'))
    np.save(os.path.join(artifacts_dir, 'X_test.npy'), X_test)
    np.save(os.path.join(artifacts_dir, 'y_test.npy'), y_test)

    print("\n[4/5] Evaluating Forecast Accuracy vs Baselines on Held-Out Test Set...")
    num_clients = len(client_cols)
    num_samples = len(X_test)
    y_pred_scaled = model.predict(X_test, verbose=0)

    y_test_true = inverse_transform_targets(y_test, scaler, num_clients)
    y_pred_model = inverse_transform_targets(y_pred_scaled, scaler, num_clients)
    y_pred_yesterday = inverse_transform_targets(X_test[:, :, :num_clients], scaler, num_clients)

    # Last Week Baseline (t-168)
    y_pred_last_week = np.zeros_like(y_test_true)
    for i in range(num_samples):
        past_idx = max(0, i - 168)
        y_pred_last_week[i] = inverse_transform_targets(X_test[past_idx:past_idx+1, :, :num_clients], scaler, num_clients)[0]

    records = []
    for c_idx, client_name in enumerate(client_cols):
        yt = y_test_true[:, :, c_idx]
        m_model = calculate_metrics(yt, y_pred_model[:, :, c_idx])
        m_yest = calculate_metrics(yt, y_pred_yesterday[:, :, c_idx])
        m_week = calculate_metrics(yt, y_pred_last_week[:, :, c_idx])

        records.append({
            'client': client_name,
            'model_mae': m_model['MAE'], 'model_rmse': m_model['RMSE'], 'model_smape': m_model['sMAPE'],
            'yesterday_mae': m_yest['MAE'], 'yesterday_rmse': m_yest['RMSE'], 'yesterday_smape': m_yest['sMAPE'],
            'last_week_mae': m_week['MAE'], 'last_week_rmse': m_week['RMSE'], 'last_week_smape': m_week['sMAPE'],
        })

    df_metrics = pd.DataFrame(records)
    avg_row = {'client': 'AVERAGE'}
    for col in df_metrics.columns:
        if col != 'client':
            avg_row[col] = df_metrics[col].mean()
    df_metrics = pd.concat([df_metrics, pd.DataFrame([avg_row])], ignore_index=True)
    df_metrics.to_csv(os.path.join(artifacts_dir, 'metrics.csv'), index=False)

    print("\n" + "="*60)
    print("TEST METRICS (UNSCALED AVERAGE)")
    print("="*60)
    avg_data = df_metrics[df_metrics['client'] == 'AVERAGE'].iloc[0]
    print(f"LSTM-Attention Model -> MAE: {avg_data['model_mae']:.2f} | RMSE: {avg_data['model_rmse']:.2f} | sMAPE: {avg_data['model_smape']:.2f}%")
    print(f"Yesterday (t-24)     -> MAE: {avg_data['yesterday_mae']:.2f} | RMSE: {avg_data['yesterday_rmse']:.2f} | sMAPE: {avg_data['yesterday_smape']:.2f}%")
    print(f"Last Week (t-168)    -> MAE: {avg_data['last_week_mae']:.2f} | RMSE: {avg_data['last_week_rmse']:.2f} | sMAPE: {avg_data['last_week_smape']:.2f}%")
    print("="*60)

    print("\n[5/5] Computing Anomaly Threshold & Running Synthetic Injection Evaluation...")
    y_val_pred_scaled = model.predict(X_val, verbose=0)
    y_val_true = inverse_transform_targets(y_val, scaler, num_clients)
    y_val_pred = inverse_transform_targets(y_val_pred_scaled, scaler, num_clients)

    val_residuals = np.abs(y_val_true - y_val_pred)
    threshold = np.mean(val_residuals) + 3 * np.std(val_residuals)
    joblib.dump(threshold, os.path.join(artifacts_dir, 'threshold.pkl'))
    print(f"Validation Residual Mean + 3*Std Threshold: {threshold:.2f} kW")

    # Synthetic Injection Evaluation (Spikes & Drops)
    y_test_injected = y_test_true.copy()
    synthetic_labels = np.zeros(y_test_true.shape, dtype=int)
    num_anomalies = int(0.02 * y_test_true.size)
    flat_indices = np.random.choice(y_test_true.size, size=num_anomalies, replace=False)

    for idx in flat_indices:
        unflat_idx = np.unravel_index(idx, y_test_true.shape)
        synthetic_labels[unflat_idx] = 1
        y_test_injected[unflat_idx] = (y_test_true[unflat_idx] * 3.5 + 50.0) if np.random.rand() > 0.5 else 0.0

    test_residuals = np.abs(y_test_injected - y_pred_model)
    predicted_anomalies = (test_residuals > threshold).astype(int)

    precision = precision_score(synthetic_labels.flatten(), predicted_anomalies.flatten())
    recall = recall_score(synthetic_labels.flatten(), predicted_anomalies.flatten())
    f1 = f1_score(synthetic_labels.flatten(), predicted_anomalies.flatten())

    print(f"Anomaly Detection -> Precision: {precision:.4f} | Recall: {recall:.4f} | F1-Score: {f1:.4f}")
    print("\nPipeline finished successfully! Artifacts stored in artifacts/\n")

if __name__ == '__main__':
    main()


[1/5] Loading and Preprocessing Electricity Dataset...

[2/5] Creating Sliding Window Sequences (24h Input -> 24h Horizon)...
X_train shape: (24498, 24, 24), y_train shape: (24498, 24, 20)
X_test shape:  (5213, 24, 24), y_test shape:  (5213, 24, 20)

[3/5] Building & Training LSTM + Multi-Head Attention Model...
Epoch 1/30
383/383 ━━━━━━━━━━━━━━━━━━━━ 56s 134ms/step - loss: 0.0297 - mae: 0.1014 - val_loss: 0.0048 - val_mae: 0.0524
Epoch 2/30
383/383 ━━━━━━━━━━━━━━━━━━━━ 81s 132ms/step - loss: 0.0046 - mae: 0.0524 - val_loss: 0.0042 - val_mae: 0.0492
Epoch 3/30
383/383 ━━━━━━━━━━━━━━━━━━━━ 81s 130ms/step - loss: 0.0040 - mae: 0.0482 - val_loss: 0.0035 - val_mae: 0.0444
Epoch 4/30
383/383 ━━━━━━━━━━━━━━━━━━━━ 53s 140ms/step - loss: 0.0036 - mae: 0.0456 - val_loss: 0.0033 - val_mae: 0.0429
Epoch 5/30
383/383 ━━━━━━━━━━━━━━━━━━━━ 50s 130ms/step - loss: 0.0033 - mae: 0.0441 - val_loss: 0.0031 - val_mae: 0.0414
Epoch 6/30
383/383 ━━━━━━━━━━━━━━━━━━━━ 50s 130ms/step - loss: 0.0031 - mae: 0.0

In [4]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 31.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 85.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [streamlit]


In [9]:
import shutil
from google.colab import files

# Zip the artifacts folder
shutil.make_archive('artifacts', 'zip', '/content/artifacts')

# Trigger browser download
files.download('artifacts.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>